# AIME — SD 3.5 Generation on Google Colab

Runs SD 3.5 Large / Medium / Turbo on the 81 image-compatible AIME prompts. Built for a Colab **T4 (15GB VRAM)** — the local 8GB laptop GPU couldn't fit the 8B-param transformer even with CPU offload (tested: offload makes it impractically slow, hours per image).

**Fix applied vs the local notebook**: the transformer is loaded 4-bit quantized (NF4) and the T5-XXL text encoder is dropped (`text_encoder_3=None`). This isn't just a Colab-VRAM workaround — even 15GB is tight for full fp16 (transformer ~16GB + T5 ~10GB + CLIP + VAE > 15GB), so the same fix is needed here too. Net effect: everything fits directly on-GPU with no offloading, so generation runs at native GPU speed.

## Before running
1. **Runtime → Change runtime type → T4 GPU** (or better, if you have Colab Pro)
2. Upload `prompts/all_prompts.xlsx` (from the `AIME-public` repo) to your Google Drive, e.g. `MyDrive/AIME/all_prompts.xlsx` — update `PROMPTS_XLS` below if you put it elsewhere
3. Add your HuggingFace token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar) — needs read access to the gated `stabilityai/stable-diffusion-3.5-*` repos (accept the licenses on huggingface.co first)
4. Set `MODEL_KEY` in the config cell below: `'turbo'` first (fastest, good smoke test), then `'medium'`, then `'large'`

## After running
Output is written to your Drive under `MyDrive/AIME/dataset/media/images/{ModelName}/` and `MyDrive/AIME/dataset/labels/generated_{ModelName}.csv` — download/sync that `dataset/` folder back into your local `AIME-public/dataset/` (it merges cleanly, same folder structure as the rest of the repo).

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q -U diffusers transformers accelerate bitsandbytes huggingface_hub pandas openpyxl tqdm

In [ ]:
# ── Mount Drive ───────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Configuration (edit these) ────────────────────────────────────────────────
import os
from google.colab import userdata

# Choose one: 'large' | 'medium' | 'turbo' -- run turbo first as a smoke test
MODEL_KEY = 'turbo'

MODEL_MAP = {
    'large'  : ('stabilityai/stable-diffusion-3.5-large',       'SD35_Large'),
    'medium' : ('stabilityai/stable-diffusion-3.5-medium',      'SD35_Medium'),
    'turbo'  : ('stabilityai/stable-diffusion-3.5-large-turbo', 'SD35_Turbo'),
}
HF_MODEL_ID, MODEL_LABEL = MODEL_MAP[MODEL_KEY]
HF_TOKEN = userdata.get('HF_TOKEN')  # Colab secret -- key icon in left sidebar

IMAGES_PER_PROMPT   = 1
NUM_INFERENCE_STEPS = 28   # 28 for large/medium; 4 for turbo (forced below)
GUIDANCE_SCALE      = 7.5  # ignored by turbo (CFG=0)

DRIVE_ROOT  = '/content/drive/MyDrive/AIME'
PROMPTS_XLS = f'{DRIVE_ROOT}/all_prompts.xlsx'
OUTPUT_DIR  = f'{DRIVE_ROOT}/dataset/media/images/{MODEL_LABEL}'
INDEX_CSV   = f'{DRIVE_ROOT}/dataset/labels/generated_{MODEL_LABEL}.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(INDEX_CSV), exist_ok=True)

print(f'Model   : {HF_MODEL_ID}')
print(f'Label   : {MODEL_LABEL}')
print(f'Output  : {OUTPUT_DIR}')

In [ ]:
# ── Load prompts ─────────────────────────────────────────────────────────────
import pandas as pd

df_all = pd.read_excel(PROMPTS_XLS, sheet_name='All Prompts (100)')
df_all.columns = df_all.columns.str.strip()

df_img = df_all[df_all['Modality'].isin(['Image', 'Both'])].copy().reset_index(drop=True)

print(f'Total prompts    : {len(df_all)}')
print(f'Image-compatible : {len(df_img)}')
print(df_img.groupby(['Type','User Profile']).size().to_string())

In [ ]:
# ── Load model (4-bit quantized transformer, no T5-XXL) ────────────────────────
import torch
from diffusers import StableDiffusion3Pipeline, SD3Transformer2DModel, BitsAndBytesConfig
from huggingface_hub import login

if HF_TOKEN:
    login(HF_TOKEN)

# The 8B-param transformer (shared by Large and Large-Turbo) is ~16GB in fp16 --
# too big even for a 15GB T4 once you add the other components. Load it 4-bit
# quantized (NF4) so it sits directly on-GPU with no offloading. Also drop the
# T5-XXL text encoder (text_encoder_3, ~10GB fp16 alone) -- AIME prompts already
# get truncated to its 256-token limit anyway, so CLIP-L/CLIP-G alone is enough.
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
transformer = SD3Transformer2DModel.from_pretrained(
    HF_MODEL_ID,
    subfolder='transformer',
    quantization_config=nf4_config,
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)

pipe = StableDiffusion3Pipeline.from_pretrained(
    HF_MODEL_ID,
    transformer=transformer,
    text_encoder_3=None,
    tokenizer_3=None,
    torch_dtype=torch.float16,
    token=HF_TOKEN,
)
pipe = pipe.to('cuda')
pipe.vae.enable_tiling()
pipe.set_progress_bar_config(disable=True)
print(f'{MODEL_LABEL} loaded on {pipe.device}')

In [ ]:
# ── Generation helpers ───────────────────────────────────────────────────────
import hashlib, datetime

def make_filename(prompt_id, prompt_type, index=0):
    ts = datetime.datetime.now().strftime('%Y%m%d%H%M%S%f')
    h  = hashlib.sha256(f'{prompt_id}_{index}_{ts}'.encode()).hexdigest()[:10]
    t  = 'amb' if prompt_type == 'Ambiguous' else 'exp'
    return f'P{prompt_id:03d}_{t}_{h}.png'

print('Helpers ready.')

In [ ]:
# ── Resume: load already-generated index ─────────────────────────────────────
if os.path.exists(INDEX_CSV):
    df_done  = pd.read_csv(INDEX_CSV)
    done_ids = set(zip(df_done['prompt_type'], df_done['prompt_id'], df_done['img_index']))
    print(f'Resuming: {len(df_done)} images already generated.')
else:
    df_done  = pd.DataFrame()
    done_ids = set()
    print('Starting fresh.')

In [ ]:
# ── Main generation loop ─────────────────────────────────────────────────────
from tqdm.notebook import tqdm

_guidance = 0.0 if MODEL_KEY == 'turbo' else GUIDANCE_SCALE
_steps    = 4   if MODEL_KEY == 'turbo' else NUM_INFERENCE_STEPS

records = []

for _, row in tqdm(df_img.iterrows(), total=len(df_img), desc=MODEL_LABEL):
    pid         = int(row['ID'])
    ptype       = row['Type']
    prompt_text = str(row['Prompt Text'])

    for img_idx in range(IMAGES_PER_PROMPT):
        if (ptype, pid, img_idx) in done_ids:
            continue

        status = 'ok'
        fname  = ''
        try:
            with torch.no_grad():
                result = pipe(
                    prompt                = prompt_text,
                    num_inference_steps   = _steps,
                    guidance_scale        = _guidance,
                    num_images_per_prompt = 1,
                )
            image = result.images[0]
            fname = make_filename(pid, ptype, img_idx)
            image.save(os.path.join(OUTPUT_DIR, fname))
            done_ids.add((ptype, pid, img_idx))
            torch.cuda.empty_cache()
        except Exception as e:
            print(f'  Error P{pid:03d}: {str(e)[:200]}')
            status = 'error'

        records.append({
            'file_path'   : f'images/{MODEL_LABEL}/{fname}' if fname else '',
            'prompt_id'   : pid,
            'img_index'   : img_idx,
            'prompt_type' : ptype,
            'user_profile': row['User Profile'],
            'modality'    : row['Modality'],
            'category'    : row['Category'],
            'language'    : row['Language'],
            'status'      : status,
            'model'       : MODEL_LABEL,
            'prompt_text' : prompt_text,
        })

        # Persist after every image -- Colab sessions can disconnect, don't lose progress
        df_new = pd.DataFrame(records)
        df_out = pd.concat([df_done, df_new], ignore_index=True) if not df_done.empty else df_new
        df_out.to_csv(INDEX_CSV, index=False)

ok  = (df_out['status'] == 'ok').sum()
err = (df_out['status'] == 'error').sum()
print(f'\nDone. Generated: {ok} | Errors: {err}')
print(f'Index: {INDEX_CSV}')

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────
df_results = pd.read_csv(INDEX_CSV)
print(f'=== {MODEL_LABEL} Generation Summary ===')
print(df_results.groupby(['prompt_type','status']).size().unstack(fill_value=0).to_string())
print(f"\nTotal images saved: {(df_results['status']=='ok').sum()} / {len(df_results)}")